# Attaching Tools with LLM

In [1]:
from langchain_groq import ChatGroq

In [19]:
llm = ChatGroq(model="openai/gpt-oss-120b")
llm.__dict__

{'name': None,
 'cache': None,
 'verbose': False,
 'callbacks': None,
 'tags': None,
 'metadata': None,
 'custom_get_token_ids': None,
 'rate_limiter': None,
 'disable_streaming': False,
 'output_version': None,
 'profile': {'max_input_tokens': 131072,
  'max_output_tokens': 32768,
  'image_inputs': False,
  'audio_inputs': False,
  'video_inputs': False,
  'image_outputs': False,
  'audio_outputs': False,
  'video_outputs': False,
  'reasoning_output': True,
  'tool_calling': True},
 'client': <groq.resources.chat.completions.Completions at 0x2251d536c30>,
 'async_client': <groq.resources.chat.completions.AsyncCompletions at 0x2251d5343b0>,
 'model_name': 'openai/gpt-oss-120b',
 'temperature': 0.7,
 'stop': None,
 'reasoning_format': None,
 'reasoning_effort': None,
 'model_kwargs': {},
 'groq_api_key': SecretStr('**********'),
 'groq_api_base': None,
 'groq_proxy': None,
 'request_timeout': None,
 'max_retries': 2,
 'streaming': False,
 'n': 1,
 'max_tokens': None,
 'service_tier': '

In [3]:
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'User asks: "What is the capital of France?" Simple factual. Provide answer: Paris.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 78, 'total_tokens': 116, 'completion_time': 0.086032044, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.00322294, 'prompt_tokens_details': None, 'queue_time': 0.004690034, 'total_time': 0.089254984}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4648-b90a-7082-9d4b-394be6b7f0ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 38, 'total_tokens': 116, 'output_token_details': {'reasoning': 20}})

In [4]:
llm.invoke("What is the sum of 5 and 3?")

AIMessage(content='The sum of 5 and 3 is **8**.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: sum of 5 and 3 = 8. No policy issues. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 82, 'total_tokens': 131, 'completion_time': 0.101821213, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.003326108, 'prompt_tokens_details': None, 'queue_time': 0.004272553, 'total_time': 0.105147321}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8db49de948', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4649-4a45-7ac1-8203-9ea62d07ed7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 49, 'total_tokens': 131, 'output_token_details': {'reasoning': 27}})

### creating tool

In [5]:
from langchain_core.tools import tool

@tool
def add_tool(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

In [6]:
print(add_tool.name)
print(add_tool.description)
print(add_tool.args)
print(add_tool.args_schema.model_json_schema())

add_tool
Adds two numbers together.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
{'description': 'Adds two numbers together.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add_tool', 'type': 'object'}


In [11]:
result = add_tool.invoke({"a": 5, "b": 3})
print("the sum is", result)

the sum is 8


### binding tool

In [18]:
llm_with_tool = llm.bind_tools([add_tool])
llm_with_tool.__dict__

{'name': None,
 'bound': ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002251D535FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002251D5364B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None),
 'kwargs': {'tools': [{'type': 'function',
    'function': {'name': 'add_tool',
     'description': 'Adds two numbers together.',
     'parameters': {'properties': {'a': {'type': 'integer'},
       'b': {'type': 'integer'}},
      'required': ['a', 'b'],
      'type': 'object'}}}]},
 'config': {},
 'config_factories': [],
 'custom_input_type': None,
 'custom_output_type': None}

### using llm with tool

In [23]:
result = llm_with_tool.invoke("plz use tool to findout that What is the sum of 5 and 3?")
result.__dict__

{'content': '',
 'additional_kwargs': {'reasoning_content': 'The user asks: "plz use tool to findout that What is the sum of 5 and 3?" We should use the provided tool add_tool to compute 5+3. Then answer.',
  'tool_calls': [{'id': 'fc_3accbb94-34b4-43c5-8290-2cafe26996e9',
    'function': {'arguments': '{"a":5,"b":3}', 'name': 'add_tool'},
    'type': 'function'}]},
 'response_metadata': {'token_usage': {'completion_tokens': 77,
   'prompt_tokens': 141,
   'total_tokens': 218,
   'completion_time': 0.173981337,
   'completion_tokens_details': {'reasoning_tokens': 43},
   'prompt_time': 0.00562275,
   'prompt_tokens_details': None,
   'queue_time': 0.004682972,
   'total_time': 0.179604087},
  'model_name': 'openai/gpt-oss-120b',
  'system_fingerprint': 'fp_a09bde29de',
  'service_tier': 'on_demand',
  'finish_reason': 'tool_calls',
  'logprobs': None,
  'model_provider': 'groq'},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--019e4652-8419-74a0-b48b-c56020e94db3-0',
 'tool_calls': [{'na

In [ ]:
# content will be empty when we will use tool calls, 
# because the response will be generated by the tool calls, 
# so the content will be empty and the tool_calls will have the response from the tool calls.
result.tool_calls

[{'name': 'add_tool',
  'args': {'a': 5, 'b': 3},
  'id': 'fc_3accbb94-34b4-43c5-8290-2cafe26996e9',
  'type': 'tool_call'}]

### using tool

In [30]:
result.tool_calls[0]['args']

{'a': 5, 'b': 3}

In [31]:
add_tool.invoke(result.tool_calls[0]['args'])

8

In [34]:
result.tool_calls[0]

{'name': 'add_tool',
 'args': {'a': 5, 'b': 3},
 'id': 'fc_3accbb94-34b4-43c5-8290-2cafe26996e9',
 'type': 'tool_call'}

In [ ]:
# here we are passing the whole tool call object to the invoke method,
# because the invoke method can also accept the tool call object and
# it will extract the args from the tool call object and pass it to the function.
# and return the response wraped in the ToolMessage object.
add_tool.invoke(result.tool_calls[0])

ToolMessage(content='8', name='add_tool', tool_call_id='fc_3accbb94-34b4-43c5-8290-2cafe26996e9')

### finalizing everything

In [ ]:
messages = []
msg = llm_with_tool.invoke("plz use tool to findout that What is the sum of 5 and 3?")
msg


AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks to use tool to find out sum of 5 and 3. Use add_tool.', 'tool_calls': [{'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'function': {'arguments': '{"a":5,"b":3}', 'name': 'add_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 141, 'total_tokens': 197, 'completion_time': 0.12233341, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.006165065, 'prompt_tokens_details': None, 'queue_time': 0.005067297, 'total_time': 0.128498475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8db49de948', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4659-4cb8-7563-a431-9409618cd04c-0', tool_calls=[{'name': 'add_tool', 'args': {'a': 5, 'b': 3}, 'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

In [39]:
messages.append(msg)
messages

[AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks to use tool to find out sum of 5 and 3. Use add_tool.', 'tool_calls': [{'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'function': {'arguments': '{"a":5,"b":3}', 'name': 'add_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 141, 'total_tokens': 197, 'completion_time': 0.12233341, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.006165065, 'prompt_tokens_details': None, 'queue_time': 0.005067297, 'total_time': 0.128498475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8db49de948', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4659-4cb8-7563-a431-9409618cd04c-0', tool_calls=[{'name': 'add_tool', 'args': {'a': 5, 'b': 3}, 'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [40]:
tool_msg = add_tool.invoke(msg.tool_calls[0])
tool_msg

ToolMessage(content='8', name='add_tool', tool_call_id='fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e')

In [41]:
messages.append(tool_msg)
messages

[AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks to use tool to find out sum of 5 and 3. Use add_tool.', 'tool_calls': [{'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'function': {'arguments': '{"a":5,"b":3}', 'name': 'add_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 141, 'total_tokens': 197, 'completion_time': 0.12233341, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.006165065, 'prompt_tokens_details': None, 'queue_time': 0.005067297, 'total_time': 0.128498475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8db49de948', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4659-4cb8-7563-a431-9409618cd04c-0', tool_calls=[{'name': 'add_tool', 'args': {'a': 5, 'b': 3}, 'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [42]:
msg = llm_with_tool.invoke(messages)
msg

AIMessage(content='The sum of 5 and 3 is **8**.', additional_kwargs={'reasoning_content': 'We have a simple addition. The user asked to add 5 and 3. We computed 8. Now respond.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 151, 'total_tokens': 199, 'completion_time': 0.101840862, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005837907, 'prompt_tokens_details': None, 'queue_time': 0.004818441, 'total_time': 0.107678769}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e465a-f93c-7832-9109-8e625ee99e12-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 48, 'total_tokens': 199, 'output_token_details': {'reasoning': 26}})

In [43]:
messages.append(msg)
messages

[AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks to use tool to find out sum of 5 and 3. Use add_tool.', 'tool_calls': [{'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'function': {'arguments': '{"a":5,"b":3}', 'name': 'add_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 141, 'total_tokens': 197, 'completion_time': 0.12233341, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.006165065, 'prompt_tokens_details': None, 'queue_time': 0.005067297, 'total_time': 0.128498475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8db49de948', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4659-4cb8-7563-a431-9409618cd04c-0', tool_calls=[{'name': 'add_tool', 'args': {'a': 5, 'b': 3}, 'id': 'fc_c18820f3-9d1f-4fbb-9538-aaa8c29f9c3e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [45]:
print(messages[-1].content)

The sum of 5 and 3 is **8**.


In [46]:
from IPython.display import display, Markdown
display(Markdown(messages[-1].content))

The sum of 5 and 3 is **8**.